# Data Science and Analytics Exam Preparation

This notebook contains analysis of three different datasets:
1. NFL Scoring Analysis (2010-2019)
2. Umbrella Sales Analysis
3. Fortune 500 Market Capitalization and Profit Analysis
4. EJB Data Analysis

Let's start by importing the necessary libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Set style for better visualizations
plt.style.use('seaborn')
sns.set_palette('husl')

# For better figure resolution
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

## 1. NFL Scoring Analysis (2010-2019)

We'll analyze how scoring distribution has changed over the seasons and examine individual team trends.

In [3]:
# Read the NFL scoring data
nfl_df = pd.read_csv('ScoringNFL.csv')
print('Data Shape:', nfl_df.shape)
nfl_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'ScoringNFL.csv'

In [ ]:
# Create a box plot to show scoring distribution across seasons
plt.figure(figsize=(15, 8))
sns.boxplot(x='Season', y='Points Scored', data=nfl_df)
plt.title('NFL Scoring Distribution by Season (2010-2019)', fontsize=14)
plt.xlabel('Season', fontsize=12)
plt.ylabel('Points Scored', fontsize=12)
plt.xticks(rotation=45)
plt.show()

# Calculate and display summary statistics for each season
seasonal_stats = nfl_df.groupby('Season')['Points Scored'].agg(['mean', 'std', 'min', 'max']).round(2)
print('\nScoring Statistics by Season:')
print(seasonal_stats)

In [ ]:
# Create sparklines for each team
# Pivot the data for team-wise analysis
team_scores = nfl_df.pivot(index='Team', columns='Season', values='Points Scored')

# Calculate trend for each team
def calculate_trend(row):
    x = np.arange(len(row))
    slope, _ = np.polyfit(x, row, 1)
    return slope

trends = team_scores.apply(calculate_trend, axis=1)
upward_trend = trends.nlargest(1)
downward_trend = trends.nsmallest(1)

# Plot sparklines for teams with strongest trends
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6))

# Upward trend
team_up = upward_trend.index[0]
ax1.plot(team_scores.columns, team_scores.loc[team_up], 'b-')
ax1.set_title(f'Upward Trend: {team_up}')

# Downward trend
team_down = downward_trend.index[0]
ax2.plot(team_scores.columns, team_scores.loc[team_down], 'r-')
ax2.set_title(f'Downward Trend: {team_down}')

plt.tight_layout()
plt.show()

## 2. Umbrella Sales Analysis

We'll analyze quarterly sales data and investigate seasonality patterns.

In [ ]:
# Read the Umbrella sales data
umbrella_df = pd.read_csv('Umbrella.csv')
print('Data Shape:', umbrella_df.shape)
umbrella_df.head()

In [ ]:
# Calculate 4-period moving average
umbrella_df['Moving_Average'] = umbrella_df['Sales (Thousands $)'].rolling(window=4).mean()

# Create time series plot with moving average
plt.figure(figsize=(15, 8))
plt.plot(range(len(umbrella_df)), umbrella_df['Sales (Thousands $)'], label='Actual Sales')
plt.plot(range(len(umbrella_df)), umbrella_df['Moving_Average'], label='4-Period Moving Average', linewidth=2)
plt.title('Umbrella Sales with 4-Period Moving Average', fontsize=14)
plt.xlabel('Time Period', fontsize=12)
plt.ylabel('Sales (Thousands $)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Create seasonal plot
plt.figure(figsize=(15, 8))
for year in umbrella_df['Year'].unique():
    year_data = umbrella_df[umbrella_df['Year'] == year]
    plt.plot(year_data['Quarter'], year_data['Sales (Thousands $)'], marker='o', label=f'Year {year}')

plt.title('Seasonal Pattern of Umbrella Sales', fontsize=14)
plt.xlabel('Quarter', fontsize=12)
plt.ylabel('Sales (Thousands $)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

## 3. Fortune 500 Market Capitalization and Profit Analysis

We'll analyze the relationship between market capitalization and profit, with a focus on the healthcare sector.

In [ ]:
# Read the Fortune 500 data
fortune_df = pd.read_csv('Fortune500Sector.csv')
print('Data Shape:', fortune_df.shape)
fortune_df.head()

In [ ]:
# Create scatter plot with healthcare sector highlighted
plt.figure(figsize=(15, 10))

# Plot non-healthcare sectors in gray
non_healthcare = fortune_df[fortune_df['Sector'] != 'Healthcare']
plt.scatter(non_healthcare['Profits ($ millions)'], 
           non_healthcare['Market Capitalization ($ millions)'],
           c='gray', alpha=0.5, label='Other Sectors')

# Plot healthcare sector
healthcare = fortune_df[fortune_df['Sector'] == 'Healthcare']
plt.scatter(healthcare['Profits ($ millions)'], 
           healthcare['Market Capitalization ($ millions)'],
           c='blue', alpha=0.7, label='Healthcare')

# Add trendline for healthcare sector
z = np.polyfit(healthcare['Profits ($ millions)'], 
               healthcare['Market Capitalization ($ millions)'], 1)
p = np.poly1d(z)
plt.plot(healthcare['Profits ($ millions)'], 
         p(healthcare['Profits ($ millions)']), 
         'r--', label='Healthcare Trendline')

plt.title('Market Capitalization vs. Profit by Sector', fontsize=14)
plt.xlabel('Profits ($ millions)', fontsize=12)
plt.ylabel('Market Capitalization ($ millions)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

# Calculate correlation for healthcare sector
correlation = healthcare['Profits ($ millions)'].corr(healthcare['Market Capitalization ($ millions)'])
print(f'\nCorrelation coefficient for Healthcare sector: {correlation:.3f}')

## 4. EJB Data Analysis

We'll analyze the Category and New Customer variables, as well as investigate missing data patterns in Product Satisfaction Rating.

In [ ]:
# Read the EJB data
ejb_df = pd.read_csv('EJB.csv')
print('Data Shape:', ejb_df.shape)
ejb_df.head()

In [ ]:
# Category relative frequency distribution
category_dist = ejb_df['Category'].value_counts(normalize=True)
print('Category Distribution:')
print(category_dist)

# Visualize Category distribution
plt.figure(figsize=(10, 6))
category_dist.plot(kind='bar')
plt.title('Relative Frequency Distribution of Categories')
plt.xlabel('Category')
plt.ylabel('Relative Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# New Customer relative frequency distribution
# Relabel the values
ejb_df['Customer_Type'] = ejb_df['New Customer'].map({'No': 'Existing', 'Yes': 'New'})
customer_dist = ejb_df['Customer_Type'].value_counts(normalize=True)
print('\nCustomer Type Distribution:')
print(customer_dist)

# Visualize Customer Type distribution
plt.figure(figsize=(8, 6))
customer_dist.plot(kind='bar')
plt.title('Relative Frequency Distribution of Customer Types')
plt.xlabel('Customer Type')
plt.ylabel('Relative Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze Product Satisfaction Rating
# Calculate percentage of missing values
missing_pct = (ejb_df['Product Satisfaction Rating'].isna().sum() / len(ejb_df)) * 100
print(f'Percentage of missing Product Satisfaction Rating: {missing_pct:.2f}%')

# Distribution for non-missing values
non_missing_flavor_dist = ejb_df[ejb_df['Product Satisfaction Rating'].notna()]['Flavor'].value_counts(normalize=True)
print('\nFlavor distribution for records with Product Satisfaction Rating:')
print(non_missing_flavor_dist)

# Distribution for missing values
missing_flavor_dist = ejb_df[ejb_df['Product Satisfaction Rating'].isna()]['Flavor'].value_counts(normalize=True)
print('\nFlavor distribution for records without Product Satisfaction Rating:')
print(missing_flavor_dist)

# Visualize both distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

non_missing_flavor_dist.plot(kind='bar', ax=ax1, title='Flavors (With Ratings)')
missing_flavor_dist.plot(kind='bar', ax=ax2, title='Flavors (Without Ratings)')

ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()